In [ ]:
import os
import requests

In [ ]:
# Add API KEY to HEADER
BASE_URL = "http://localhost:8000"
API_KEY = os.environ.get("NORMAL_USER_API_KEY", "your_api_key_here") 
session = requests.Session()
session.headers.update({"X-API-Key": API_KEY})

In [ ]:
#Helper Functions
import io
from pathlib import Path
import pandas as pd

SEEDS = Path.cwd().parent / "seeds" / "hypercover"

def upload_csv(table_name: str, rel_path: str):
    path = SEEDS / rel_path
    with open(path, "rb") as f:
        r = session.post(
            f"{BASE_URL}/data/upload/{table_name}",
            files={"upload_file": (path.name, f, "text/csv")},
        )
    print(r.status_code, r.text)
    r.raise_for_status()


def inspect(table_name: str) -> pd.DataFrame:
    r = session.get(f"{BASE_URL}/data/{table_name}")
    r.raise_for_status()
    if "currently empty" in r.text:
        print(f"GET /data/{table_name}: {r.text}")
        return pd.DataFrame()
    df = pd.read_csv(io.StringIO(r.text), sep=";")
    print(f"GET /data/{table_name}: {len(df)} rows")
    return df

In [ ]:
# Check my user has writer role
r = session.get(f"{BASE_URL}/auth/me")
r.raise_for_status()
me = r.json()
me

In [ ]:
CATALOGS = [
    ("unit", "00_catalogs/01_units.csv"),
    ("variable", "00_catalogs/02_variables.csv"),
    ("crop_type", "00_catalogs/03_crop_types.csv"),
    ("treatment", "00_catalogs/04_treatments.csv"),
]

before = {table: len(inspect(table)) for table, _ in CATALOGS}

results = []
for table, rel_path in CATALOGS:
    try:
        upload_csv(table, rel_path)
        results.append({"table": table, "file": rel_path, "result": "inserted"})
    except requests.HTTPError as err:
        code = err.response.status_code
        if code == 409:
            results.append({"table": table, "file": rel_path, "result": "already present (409)"})
        else:
            results.append({"table": table, "file": rel_path, "result": f"failed ({code})"})

counts = {table: len(inspect(table)) for table, _ in CATALOGS}

report = pd.DataFrame(results)
report["rows_now"] = report["table"].map(counts)
report

In [ ]:
inspect("treatment")